# FemHealth AI — Detecção de PCOS (Síndrome dos Ovários Policísticos)
**Tech Challenge FIAP PosTech — Fase 1**

---

## Contexto Clínico

A Síndrome dos Ovários Policísticos (PCOS/SOP) é a endocrinopatia mais comum em mulheres em idade reprodutiva, afetando entre 6% e 15% dessa população. Caracteriza-se por disfunção ovulatória, hiperandrogenismo e morfologia policística dos ovários.

**Impacto clínico:**
- Principal causa de infertilidade anovulatória
- Risco aumentado de diabetes tipo 2, síndrome metabólica e doenças cardiovasculares
- Diagnóstico frequentemente tardio — muitas mulheres passam anos sem diagnóstico correto

**Dataset:** PCOS Disease Dataset (Kaggle — Prasoon Kottarathil) — arquivo `PCOS_infertility.csv`

| Atributo | Valor |
|---|---|
| Amostras | 541 |
| Features | 3 (biomarkers hormonais) |
| Variável alvo | `PCOS (Y/N)` — 0 = Sem PCOS, 1 = Com PCOS |
| Dado inconsistente | Linha 307: AMH = `'a'` (substituído por NaN e imputado) |

**Biomarkers utilizados:**
- **AMH (ng/mL):** Anti-Müllerian Hormone — produzido pelos folículos antrais. Valores elevados indicam maior reserva ovariana, característica central da PCOS.
- **beta-HCG I e II (mIU/mL):** Medições de gonadotrofina coriônica humana coletadas em momentos distintos — utilizadas no contexto de diagnóstico diferencial de infertilidade.

**Objetivo:** Construir e comparar modelos de classificação binária para detecção precoce de PCOS, priorizando **Recall** — um falso negativo (PCOS não detectado) atrasa o tratamento e pode agravar as consequências reprodutivas e metabólicas.

## 1. Configuração do Ambiente

In [ ]:
# Instala todas as dependências necessárias
!pip install numpy pandas matplotlib seaborn shap joblib scikit-learn jinja2 -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)
DATA_PATH = Path('PCOS_infertility.csv')

print('Ambiente configurado ✓')

## 2. Carregamento e Inspeção dos Dados

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

# Limpeza dos nomes das colunas (espacos extras)
df_raw.columns = df_raw.columns.str.strip()

print(f'Shape original : {df_raw.shape}')
print(f'Colunas: {list(df_raw.columns)}')
df_raw.head(10)

In [ ]:
df_raw.info()
print(f'\nValores nulos por coluna:')
print(df_raw.isnull().sum())

In [ ]:
df_raw.describe()

**Observação sobre qualidade dos dados:**
- A coluna `AMH(ng/mL)` foi lida como `object` em vez de `float64` — indica a presença de um valor inválido.
- Investigação: linha 307 contém o valor `'a'` em `AMH(ng/mL)`, provavelmente por erro de digitação no prontuário.
- **Estratégia:** converter a coluna para numérico com `errors='coerce'` (transforma `'a'` em `NaN`) e imputar com a mediana no pipeline.

## 3. Pré-processamento de Dados

In [ ]:
# Remove colunas de identificacao
DROP_COLS = ['Sl. No', 'Patient File No.']
TARGET    = 'PCOS (Y/N)'

df = df_raw.drop(columns=DROP_COLS)

# Converte todas as features para numerico (transforma 'a' em NaN)
feature_cols = [c for c in df.columns if c != TARGET]
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

X = df[feature_cols]
y = df[TARGET]

print(f'Features       : {list(X.columns)}')
print(f'Shape X        : {X.shape}')
print(f'Valores NaN    : {X.isnull().sum().to_dict()}')
print(f'\nDistribuicao do target:')
print(y.value_counts().rename({0: 'Sem PCOS (0)', 1: 'Com PCOS (1)'})
)

In [ ]:
# Renomeia colunas para facilitar leitura nos graficos
rename_map = {
    'I   beta-HCG(mIU/mL)' : 'beta_HCG_I',
    'II    beta-HCG(mIU/mL)': 'beta_HCG_II',
    'AMH(ng/mL)'            : 'AMH',
}
X = X.rename(columns=rename_map)
feature_cols = list(X.columns)

df_eda = X.copy()
df_eda['PCOS']      = y.values
df_eda['diagnostico'] = y.map({0: 'Sem PCOS', 1: 'Com PCOS'}).values

print(f'Colunas finais: {feature_cols}')
X.describe()

**Estatísticas descritivas:**
- `beta_HCG_I` e `beta_HCG_II` apresentam distribuição muito assimétrica (max > 32.000 mIU/mL, enquanto mediana é próxima de 2 mIU/mL) — forte presença de outliers.
- `AMH` tem range mais restrito (0 – ~25 ng/mL), condizente com valores clínicos normais e elevados em PCOS.
- A assimetria dos beta-HCG é esperada: a maioria das pacientes não grávidas tem valores muito baixos, enquanto grávidas (incluídas no contexto de infertilidade) têm valores altíssimos. O `StandardScaler` e o `SimpleImputer` lidam com isso no pipeline.

## 4. Análise Exploratória de Dados (EDA)

### 4.1 Distribuição das Classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = y.value_counts().rename({0: 'Sem PCOS', 1: 'Com PCOS'})
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 3, str(val), ha='center', fontweight='bold')
axes[0].set_title('Contagem por Classe', fontsize=12)
axes[0].set_ylabel('Numero de pacientes')

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Proporcao Com / Sem PCOS', fontsize=12)

plt.suptitle('Distribuicao das Classes — PCOS Infertility Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observação:** Desbalanceamento moderado — 67,3% sem PCOS e 32,7% com PCOS. Condizente com prevalência clínica esperada. O parâmetro `stratify=y` no split preserva essa proporção nos conjuntos de treino e teste.

### 4.2 Distribuição dos Biomarkers por Classe

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = {'Sem PCOS': '#2ecc71', 'Com PCOS': '#e74c3c'}

for ax, feat in zip(axes, feature_cols):
    sns.boxplot(
        data=df_eda, x='diagnostico', y=feat,
        palette=palette, order=['Sem PCOS', 'Com PCOS'],
        ax=ax, linewidth=1.5
    )
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Distribuicao dos Biomarkers por Classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = {'Sem PCOS': '#2ecc71', 'Com PCOS': '#e74c3c'}

for ax, feat in zip(axes, feature_cols):
    sns.violinplot(
        data=df_eda, x='diagnostico', y=feat,
        palette=palette, order=['Sem PCOS', 'Com PCOS'],
        ax=ax, inner='quartile', linewidth=1.5
    )
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Violin Plots dos Biomarkers por Classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights clínicos:**
- **AMH:** Separação mais clara entre as classes — pacientes com PCOS têm valores de AMH consistentemente mais altos. Isso é esperado: ovários policísticos têm mais folículos antrais, que produzem AMH em maior quantidade. AMH > 3.4 ng/mL é frequentemente usado como critério diagnóstico.
- **beta-HCG I e II:** Grande variabilidade em ambos os grupos, com muitos outliers extremos (valores > 1.000 mIU/mL indicam possível gestação). A separação entre classes é menos clara, sugerindo menor poder discriminativo isolado.

### 4.3 Gráfico de Dispersão (Pair Plot)

In [ ]:
g = sns.pairplot(
    df_eda[feature_cols + ['diagnostico']],
    hue='diagnostico',
    palette={'Sem PCOS': '#2ecc71', 'Com PCOS': '#e74c3c'},
    plot_kws={'alpha': 0.4, 's': 20},
    diag_kind='kde'
)
g.figure.suptitle('Pair Plot — Biomarkers PCOS', y=1.02, fontsize=13, fontweight='bold')
plt.show()

### 4.4 Análise de Correlação

In [ ]:
corr = X.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap de correlacao entre features
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.3f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=1, ax=axes[0], annot_kws={'size': 11}
)
axes[0].set_title('Correlacao entre Features', fontsize=12, fontweight='bold')

# Correlacao com o target
target_corr = X.corrwith(y.astype(float)).sort_values(ascending=True)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in target_corr.values]
axes[1].barh(range(len(target_corr)), target_corr.values, color=colors, alpha=0.85)
axes[1].set_yticks(range(len(target_corr)))
axes[1].set_yticklabels(target_corr.index, fontsize=11)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlacao com o Target (Com PCOS = 1)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Correlacao de Pearson')

plt.tight_layout()
plt.show()

print('Correlacao com o target (PCOS):')
print(target_corr.sort_values(ascending=False).to_string())

**Análise de correlação:**
- `AMH` apresenta a maior correlação positiva com PCOS — confirma sua relevância clínica como biomarcador.
- `beta_HCG_I` e `beta_HCG_II` apresentam alta correlação entre si (esperado: são medições do mesmo hormônio em momentos diferentes).
- A correlação entre beta-HCG e PCOS é menor, indicando que isoladamente esses marcadores discriminam menos.
- Correlação de Pearson captura apenas relações lineares. Modelos não-lineares (Random Forest, Árvore de Decisão) podem identificar padrões além do que a correlação revela.

## 5. Pipeline de Pré-processamento

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Treino : {X_train.shape[0]} amostras ({X_train.shape[0]/len(X):.0%})')
print(f'Teste  : {X_test.shape[0]} amostras ({X_test.shape[0]/len(X):.0%})')
print(f'\nDistribuicao treino:')
print(y_train.value_counts(normalize=True).rename({0: 'Sem PCOS', 1: 'Com PCOS'}).round(3))
print(f'\nDistribuicao teste:')
print(y_test.value_counts(normalize=True).rename({0: 'Sem PCOS', 1: 'Com PCOS'}).round(3))

# Pipeline: imputacao (mediana) + normalizacao
base_steps = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
]

# Verifica o pipeline
_pipe = Pipeline(base_steps)
X_t = _pipe.fit_transform(X_train)
print(f'\nApos pipeline:')
print(f'  NaNs  : {np.isnan(X_t).sum()}')
print(f'  Media : {X_t.mean():.4f} (esperado ~0)')
print(f'  Std   : {X_t.std():.4f} (esperado ~1)')

**Design do pipeline:**
- `SimpleImputer(strategy='median')`: substitui o valor `NaN` (originalmente `'a'`) pela mediana do conjunto de treino. Mediana é preferível à média por ser robusta aos outliers extremos de beta-HCG.
- `StandardScaler`: centraliza e normaliza — essencial para Regressão Logística.
- O pipeline é **ajustado apenas no treino** e aplicado ao teste — elimina data leakage.
- `stratify=y`: garante proporção de classes preservada nos dois conjuntos.

## 6. Modelagem — Três Algoritmos Comparados

| Algoritmo | Justificativa |
|---|---|
| Regressão Logística | Baseline interpretável; coeficientes revelam direção e magnitude de cada biomarcador |
| Árvore de Decisão | Regras de decisão auditáveis (ex: se AMH > X então PCOS provável) |
| Random Forest | Ensemble robusto; feature importance nativa; melhor generalização |

In [ ]:
models = {
    'Regressao Logistica': Pipeline(
        base_steps + [('clf', LogisticRegression(max_iter=10000, random_state=RANDOM_STATE))]
    ),
    'Arvore de Decisao': Pipeline(
        base_steps + [('clf', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))]
    ),
    'Random Forest': Pipeline(
        base_steps + [('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))]
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1v  = f1_score(y_test, y_pred)
    results[name] = {
        'model'   : model,
        'y_pred'  : y_pred,
        'accuracy': acc,
        'recall'  : rec,
        'f1'      : f1v,
        'cm'      : confusion_matrix(y_test, y_pred),
    }
    print(f'{name:22s}: Acc={acc:.4f} | Recall={rec:.4f} | F1={f1v:.4f}')

## 7. Treinamento e Avaliação dos Modelos

### 7.1 Relatórios de Classificação

In [ ]:
for name, res in results.items():
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {name}')
    print(sep)
    print(classification_report(y_test, res['y_pred'], target_names=['Sem PCOS', 'Com PCOS']))

### 7.2 Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, res) in zip(axes, results.items()):
    ConfusionMatrixDisplay(res['cm'], display_labels=['Sem PCOS', 'Com PCOS']).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name, fontsize=11, fontweight='bold')
plt.suptitle('Matrizes de Confusao — Conjunto de Teste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.3 Comparação de Métricas

In [ ]:
metrics_df = pd.DataFrame({
    name: {
        'Accuracy': res['accuracy'],
        'Recall'  : res['recall'],
        'F1-Score': res['f1'],
    }
    for name, res in results.items()
}).T

print(metrics_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_df))
width = 0.25
for i, (metric, color) in enumerate(zip(
    ['Accuracy', 'Recall', 'F1-Score'],
    ['#3498db', '#e74c3c', '#2ecc71']
)):
    bars = ax.bar(x + i * width, metrics_df[metric], width,
                  label=metric, color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=8
        )
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_df.index, fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.legend()
ax.set_title('Comparacao de Modelos — PCOS', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Escolha das métricas:**
- **Recall** é a métrica prioritária: minimiza falsos negativos (paciente com PCOS não diagnosticada). Uma mulher com PCOS não detectada pode enfrentar anos de infertilidade sem tratamento adequado e maior risco de complicações metabólicas.
- **F1-Score** equilibra Precision e Recall — relevante pois falsos positivos também têm custo: exames complementares desnecessários e ansiedade desnecessária para a paciente.
- **Accuracy** pode ser enganosa com classes desbalanceadas (33% positivos). Um modelo que sempre prediz "Sem PCOS" teria accuracy de 67% — ainda assim inútil clinicamente.

### 7.4 Validação Cruzada (5-fold)

In [ ]:
print('Validacao Cruzada — 5-fold')
print('-' * 55)

for metric_name, metric_fn in [('F1-Score', 'f1'), ('Recall', 'recall')]:
    print(f'\nMetrica: {metric_name}')
    for name, res in results.items():
        scores = cross_val_score(res['model'], X, y, cv=5, scoring=metric_fn)
        print(f'  {name:22s}: {scores.mean():.4f} +/- {scores.std():.4f}')

**Interpretação:** A validação cruzada revela a estabilidade de cada modelo em diferentes partições dos dados — especialmente importante com apenas 541 amostras. Alto desvio padrão indica que o modelo é sensível ao split específico dos dados, o que é um sinal de alerta para uso em produção.

## 8. Explicabilidade dos Modelos

Em contexto clínico, explicabilidade é tão importante quanto acurácia. O profissional de saúde precisa entender **por que** o modelo fez determinada predição para aceitá-la ou questioná-la.

### 8.1 Feature Importance — Random Forest

In [ ]:
rf_clf = results['Random Forest']['model'].named_steps['clf']
importances = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if imp == importances.max() else '#3498db' for imp in importances.values]
importances.plot(kind='barh', ax=ax, color=colors)
for i, (idx, val) in enumerate(importances.items()):
    ax.text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=10)
ax.set_title('Feature Importance — Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importancia relativa')
ax.set_xlim(0, importances.max() * 1.2)
plt.tight_layout()
plt.show()

print('Feature Importance:')
print(importances.sort_values(ascending=False).to_string())

### 8.2 SHAP Values — Explicabilidade Global e Individual

In [ ]:
rf_model = results['Random Forest']['model']
clf      = rf_model.named_steps['clf']
prep     = Pipeline(rf_model.steps[:-1])

X_test_t = pd.DataFrame(prep.transform(X_test), columns=feature_cols)

explainer      = shap.TreeExplainer(clf)
shap_values_raw = explainer(X_test_t)

# RandomForest binário retorna shape (n_amostras, n_features, n_classes)
# Seleciona classe positiva (Com PCOS = índice 1)
if len(shap_values_raw.values.shape) == 3:
    shap_values = shap_values_raw[:, :, 1]
else:
    shap_values = shap_values_raw

plt.figure(figsize=(10, 5))
shap.plots.beeswarm(shap_values, max_display=10, show=False)
plt.title('SHAP Beeswarm — Impacto dos Biomarkers (Classe: Com PCOS)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall para a primeira amostra positiva (Com PCOS)
pcos_indices = np.where(y_test.values == 1)[0]
idx      = pcos_indices[0] if len(pcos_indices) > 0 else 0
pred     = rf_model.predict(X_test.iloc[[idx]])[0]
prob     = rf_model.predict_proba(X_test.iloc[[idx]])[0]
pred_lbl = 'Com PCOS' if pred == 1 else 'Sem PCOS'
real_lbl = 'Com PCOS' if y_test.iloc[idx] == 1 else 'Sem PCOS'

print(f'Biomarkers da paciente selecionada:')
print(X_test.iloc[idx].to_string())
print(f'\nPredicao : {pred_lbl} | Real: {real_lbl}')
print(f'P(Sem PCOS) = {prob[0]:.1%} | P(Com PCOS) = {prob[1]:.1%}')

plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_values[idx], max_display=10, show=False)
plt.title(f'SHAP Waterfall — Predicao: {pred_lbl} (Real: {real_lbl})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretação dos gráficos SHAP:**

**Beeswarm (visão global):** Cada ponto é uma paciente. A posição no eixo X mostra o quanto a feature empurra a predição para "Com PCOS" (positivo) ou "Sem PCOS" (negativo). A cor mostra o valor da feature (vermelho = alto, azul = baixo). Espera-se que `AMH` alto empurre para "Com PCOS".

**Waterfall (predição individual):** Parte do valor base (média do modelo no conjunto de teste) e mostra como cada biomarker desta paciente específica contribui para a predição final. Barras vermelhas aumentam o risco de PCOS; azuis diminuem. Esse nível de explicabilidade é fundamental para o médico: ele consegue ver se o AMH elevado desta paciente é o principal fator da suspeita de PCOS.

In [ ]:
# Seleciona e salva o modelo campiao (maior F1)
champion_name  = max(results, key=lambda n: results[n]['f1'])
champion_model = results[champion_name]['model']

champ_f1  = results[champion_name]['f1']
champ_rec = results[champion_name]['recall']
champ_acc = results[champion_name]['accuracy']

print(f'Modelo campiao : {champion_name}')
print(f'  F1-Score : {champ_f1:.4f}')
print(f'  Recall   : {champ_rec:.4f}')
print(f'  Accuracy : {champ_acc:.4f}')

joblib.dump(champion_model, MODELS_DIR / 'pcos_champion.pkl')
joblib.dump(feature_cols,   MODELS_DIR / 'pcos_feature_names.pkl')
print(f'\nModelo salvo em: models/pcos_champion.pkl')

## 9. Conclusões e Discussão Crítica

### Resumo dos Resultados

| Aspecto | Resultado |
|---|---|
| Dataset | PCOS Infertility — 541 amostras, 3 biomarkers |
| Dado inconsistente | AMH = `'a'` em uma linha (imputado com mediana) |
| Desbalanceamento | 67.3% Sem PCOS / 32.7% Com PCOS — manejável com `stratify` |
| Biomarker mais relevante | AMH (maior correlação e feature importance) |
| Métrica prioritária | Recall (minimização de falsos negativos) |

### O modelo pode ser utilizado na prática?

**Com ressalvas importantes, sim — como ferramenta de triagem.**

**Pontos fortes:**
- AMH é clinicamente reconhecido como um dos melhores biomarcadores isolados para PCOS
- O modelo integra automaticamente os três marcadores, ponderando-os de forma não-linear
- Explicabilidade via SHAP permite ao médico auditar cada predição

**Limitações críticas:**
1. **Apenas 3 features:** O diagnóstico de PCOS pelos Critérios de Rotterdam requer avaliação de anovulação, hiperandrogenismo e morfologia ultrassonográfica. Este modelo cobre apenas aspectos bioquímicos parciais.

2. **Dataset pequeno (541 amostras):** Insuficiente para capturar toda a variabilidade clínica da PCOS. Validação em dataset maior e multicêntrico é essencial antes de qualquer uso clínico.

3. **Beta-HCG e PCOS:** A relação entre beta-HCG e PCOS não é direta — valores altos refletem gestação. Incluir essa variável sem contextualizar o estado gestacional da paciente pode introduzir confusão.

4. **Viés de seleção:** Dados coletados em clínica de infertilidade não representam a população geral de mulheres com PCOS.

**Recomendação de uso:** Utilizar como ferramenta de **triagem de segundo nível** — após coleta de AMH e beta-HCG, o modelo sinaliza casos que devem ser priorizados para avaliação completa (ultrassom e dosagem de andrógenos). **O diagnóstico final é sempre responsabilidade do médico especialista.**

### Aviso Legal

> **Este sistema é uma ferramenta de apoio ao diagnóstico.** O diagnóstico final de PCOS requer avaliação clínica completa por ginecologista ou endocrinologista. Não substitui consulta médica, exames físicos ou avaliação especializada. **O médico sempre deve ter a palavra final.**